# This was  used for field target selection!

In [2]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.datasets
import torchvision.transforms as transforms
import random
import rasterio
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
import glob
import sys
from google.colab import drive
from rasterio.features import shapes
from shapely.geometry import shape


drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/IRP/scripts')

import geostats as gs
from helpers import tpi, multidirectional_hillshade, get_tifs, merge_dems

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# from deep-learning lectures / excercises

def set_seed(seed):
    """
    Use this to set ALL the random seeds to a fixed value and take out any randomness from cuda kernels
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False  ##uses the inbuilt cudnn auto-tuner to find the fastest convolution algorithms. -
    torch.backends.cudnn.enabled   = False

    return True

device = 'cpu'
if torch.cuda.device_count() > 0 and torch.cuda.is_available():
    print("Cuda installed! Running on GPU!")
    device = 'cuda'
else:
    print("No GPU available!")

Cuda installed! Running on GPU!


CHANGE TO NOT OVERWRITE!!

In [4]:
out_path = 'wells_only'

# Datasets

In [5]:
# change depending on what layers want to be used.
# in deployment, layers are calculated in __getitem__ becasue they are only run once
# whereas they are run in 100s of epochs in training, so recalculation is inefficient.

_BANDS_TO_LOAD = ['HILLSHADE', 'DEM']

# Bands normalised per-tile (z-score), includes non-input bands (derived in __get_item__)
_PER_TILE_NORM = {'HILLSHADE', 'DEM', 'TPI_FINE', 'TPI_COARSE'}

# AI assistance with InferenceDataset (changing from whole raster-based calculation
# to __getitem__ calculation (as discussed above))

class InferenceDataset(Dataset):
    def __init__(self, dem, tile_size=512, overlap=0.3):

        self.dem = dem
        self.shape = self.dem.shape
        self.res = 1
        self.tile_size = tile_size
        self.stride = int(tile_size * (1 - overlap))
        self.tiles = self._index_tiles()

    def _index_tiles(self):
        rows, cols = self.shape
        tiles = []
        for r in range(0, rows - self.tile_size + 1, self.stride):
            for c in range(0, cols - self.tile_size + 1, self.stride):
                tiles.append((r, c))
        return tiles

    @classmethod
    def from_array(cls, dem: np.ndarray, transform, resolution: float, tile_size=512, overlap=0.3):
        dataset = cls.__new__(cls)
        dataset.dem       = dem.astype(np.float32)
        dataset.transform = transform
        dataset.res       = resolution
        dataset.shape     = dem.shape
        dataset.tile_size = tile_size
        dataset.stride    = int(tile_size * (1 - overlap))
        dataset.tiles     = dataset._index_tiles()
        return dataset

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):
      r, c = self.tiles[idx]
      dem  = self.dem[r:r+self.tile_size, c:c+self.tile_size]

      tpi_fine   = tpi(dem, r=20)
      tpi_coarse = tpi(dem, r=50)
      hillshade  = multidirectional_hillshade(dem, self.res)

      tile = np.stack([hillshade, dem, tpi_fine, tpi_coarse], axis=0).astype(np.float32)

      for i, name in enumerate(['HILLSHADE', 'DEM', 'TPI_FINE', 'TPI_COARSE']):
          if name in _PER_TILE_NORM:
              band = tile[i]
              tile[i] = (band - band.mean()) / (band.std() + 1e-6)

      return torch.from_numpy(np.ascontiguousarray(tile)), (r, c)

In [6]:
# AI assistance with channel attention layer, was looking for a way to make
# layers more dominant (hoping it 'chooses' hillshade layer)

# basic blocks

class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

# weights input channels, learns which ones are most important
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.fc(self.avg(x).squeeze(-1).squeeze(-1))
        return x * w.unsqueeze(-1).unsqueeze(-1)

# encoder block
class Down(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_ch, out_ch),
        )

    def forward(self, x):
        return self.block(x)

# decoder block
class Up(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)

        if x.shape != skip.shape:
            x = F.pad(x, [0, skip.shape[-1] - x.shape[-1],
                           0, skip.shape[-2] - x.shape[-2]])

        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet(nn.Module):

    def __init__(self, in_channels: int = 1, base_filters: int = 32):
        super().__init__()
        f = base_filters

        self.enc1 = DoubleConv(in_channels, f)
        self.enc2 = Down(f, f * 2)
        self.enc3 = Down(f * 2, f * 4)
        self.enc4 = Down(f * 4, f * 8)

        self.bottleneck_attn = ChannelAttention(f * 16)
        self.bottleneck_drop = nn.Dropout2d(p=0.2)
        self.bottleneck = Down(f * 8, f * 16)

        self.dec4 = Up(f * 16 + f * 8, f * 8)
        self.dec3 = Up(f * 8 + f * 4, f * 4)
        self.dec2 = Up(f * 4 + f * 2, f * 2)
        self.dec1 = Up(f * 2 + f, f)

        #single logit per pixel, sigmoid applied in loop
        self.out_conv = nn.Conv2d(f, 1, kernel_size=1)

    def forward(self, x):
        #encoder anjd skips
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)

        b  = self.bottleneck(s4)
        b = self.bottleneck_attn(self.bottleneck(s4))
        b = self.bottleneck_drop(self.bottleneck_attn(self.bottleneck(s4)))

        #decoder
        x  = self.dec4(b,  s4)
        x  = self.dec3(x,  s3)
        x  = self.dec2(x,  s2)
        x  = self.dec1(x,  s1)

        return self.out_conv(x)

In [7]:
# ai assistanec with inference loop, primarily with RAM usage and hanning window
# hanning window to deal with edge artefacts in prediction (with overlap too)

def run_inference(model, dem_path, out_path, device, meta, tile_size=512, overlap=0.3, batch_size=4):
    dataset = InferenceDataset(dem_path, tile_size=tile_size, overlap=overlap)
    loader  = DataLoader(dataset, batch_size=batch_size, num_workers=4, pin_memory=True)

    rows, cols = dataset.shape

    # memory-map to disk, avoids holding two full arrays in RAM
    tmp_accum   = np.memmap('/tmp/_accum.dat',   dtype=np.float32, mode='w+', shape=(rows, cols))
    tmp_weights = np.memmap('/tmp/_weights.dat', dtype=np.float32, mode='w+', shape=(rows, cols))

    win = np.outer(np.hanning(tile_size), np.hanning(tile_size)).astype(np.float32)

    model.eval()
    with torch.no_grad():
        for tiles, (rs, cs) in loader:
            preds = torch.sigmoid(model(tiles.to(device).float())).squeeze(1)
            preds = preds.cpu().numpy()

            for pred, r, c in zip(preds, rs.numpy(), cs.numpy()):
                r, c = int(r), int(c)
                sl = np.s_[r:r+tile_size, c:c+tile_size]
                tmp_accum[sl] += pred * win
                tmp_weights[sl] += win

            del preds   # free ram before next batch

    valid = tmp_weights > 0
    tmp_accum[valid] /= tmp_weights[valid]

    prediction = (tmp_accum > 0.90).astype(np.uint8)   # uint8 for space in ram and drive

    # del up memmaps
    del tmp_accum, tmp_weights

    meta.update(dtype=np.uint8, count=1, nodata=0)
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(prediction, 1)

    return prediction

In [8]:
drive_path = '/content/drive/MyDrive/IRP/DEMTiles/MendipHillsOnly/*.tif'
paths, _, _ = get_tifs(drive_path) # returns paths, file_names, duplicates

In [10]:
dem, meta = merge_dems(paths)

In [11]:
model = UNet(in_channels=4, base_filters=64).to(device)
state_dict = torch.load('/content/drive/MyDrive/IRP/HillShadeTwoTPIs1ThresholdHatchToo.pt', map_location=device)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [12]:
pred = run_inference(model, dem, f'/content/drive/MyDrive/IRP/preds/prediction_{out_path}.tif', 'cuda', meta)

In future i should save centroids as points in a gpd shapefile or something, so i can merge them all with worrying about massive rasters etc. Then i can do comparison with the whole dataset.

In [13]:
with rasterio.open(f'/content/drive/MyDrive/IRP/preds/prediction_{out_path}.tif') as src:
    mask = src.read(1)

    geoms = [
        shape(geom)
        for geom, value in shapes(
            mask,
            mask=mask.astype(bool),
            transform=src.transform
        )
        if value == 1
    ]

gdf = gpd.GeoDataFrame(geometry=geoms, crs=src.crs)

points = gdf.centroid

In [14]:
analyser = gs.Analyser(pred, dem, points, out_path)
analyser.save()

All object metrics saved to: /content/drive/MyDrive/IRP/figs/wells_only_object_metrics.npz
